In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import os

import pandas as pd 

from time import sleep

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from pandas import ExcelWriter

import datetime

from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait



In [ ]:


# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'US NAIC' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Read data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
scriptfolder=os.getcwd()

os.chdir(scriptfolder)

# writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)




Running US NAIC Web Scraping Tool v.1.0


In [3]:

# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

regdict={

        'US NAIC 1': 'https://content.naic.org/cis_refined_results.htm?INSURANCE_TYPE=(All)&COCODE=67652&REALM=PROD',

         }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],

          'Phone - Mother company': [], 'Check': []}



processdate = now.strftime('%Y-%m-%d')




In [4]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def click_element(XPATH, Msg=None):

    for time in range(10):

        try:

            driver.find_element(By.XPATH, XPATH).click()

            if Msg!=None:

                print(f"[INFO] : - {Msg}")

            break

        except:

            print(f"[ERROR] : Retrying {time+1}/10 to click element in this page: {XPATH} ")

            sleep(1)

    else:

        raise Exception('[ERROR] : Failed to get presence for this element in this page:')





def bourange_same_length_array(sqldict) :



    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():



        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')



            sqldict[key]=sqldict[key]+empty

    return sqldict



def click_on_cookies():

    try:

        driver.find_element(By.XPATH,f'//*[@id="ctl00_g_b9bf4a64_239a_4920_92f8_bcb8705e4839"]//*/span[contains(text(),"I Accept")]').click()

    except Exception as err:

        print('[ERROR] : Failed to click "I Accept" button on the cookies banner:', err)





def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]



def switch_to_iframe() :

    element_frames = driver.find_elements(By.TAG_NAME, "iframe") # Finf all iframe. it is 03 for this page

    driver.switch_to.frame(element_frames[0]) # switch to first iframe. this frame contain other iframe


def scroll_page_bottom(driver, max_rounds=30, pause=1.2):
    # wait page ready
    WebDriverWait(driver, 20).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )

    last = -1
    for _ in range(max_rounds):
        h = driver.execute_script("""
            const e = document.scrollingElement || document.documentElement || document.body;
            e.scrollTop = e.scrollHeight;
            return e.scrollHeight;
        """)
        sleep(pause)

        now = driver.execute_script("""
            const e = document.scrollingElement || document.documentElement || document.body;
            return e.scrollHeight;
        """)
        if now == last:
            break
        last = now




In [5]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):



    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")

    driver.get('about:blank')

    driver.get(regdict[reg])

    sleep(10)

    soup=BeautifulSoup(driver.page_source, 'html.parser')

    link = driver.find_element(By.ID,'tableau').get_attribute('src')

    driver.get(link)
    
    sleep(2)
    last_height = driver.execute_script("return document.body.scrollHeight")
    sleep(2)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    
    sleep(10)
    click_element('//*[@id="download"]', 'Clisk download Bouton')

    sleep(2)

    #click_element('//*[@id="viz-viewer-toolbar-download-menu"]//*/label[contains(text(),"Crosstab")]')
    
    click_element('//*[@id="viz-viewer-toolbar-download-menu"]/div[3]')

    sleep(2)

    click_element('//*[@id="export-crosstab-options-dialog-Dialog-BodyWrapper-Dialog-Body-Id"]//*/span[contains(text(),"Company Results Phone")]')

    sleep(2)

    click_element('//*[@id="export-crosstab-options-dialog-Dialog-BodyWrapper-Dialog-Body-Id"]//*/button[contains(text(),"Download")]')

    file =  check_dowload_files(tempfolder, "Excel" )

    filePath = os.path.join(tempfolder, file)

    columns = ['Company Name', 'Web Site', 'Insurance Types', 'Company Licensed', 'Company Information', 'Cocode', 'URL_START', 'WEB_SITE'] # A definir...

    df = pd.read_excel(filePath)

    df = df.iloc[1:, 0: 8]

    df.columns = columns

    df = df.fillna("")

    df = df.reset_index(drop=True)



    print(f"[INFO] : - DataFrame '{file}' | containe = {df.shape}")

    for index, row in df.iterrows():

        info = str(row['Company Information'])

        adress = str(row['Company Information']).replace('www', 'WWW').split('WWW')[0]

        phone = ''



        if info.find('-') != -1 :

            phone = info[-13:].strip() if info[-1]=='-' else info[-12:].strip()



        

        sqldict['Name'].append(row['Company Name'].split('(NAIC')[0].strip())

        sqldict['Address_1'].append(adress.split(',')[0])



        if len(adress.split(',')) > 2:

            sqldict['Address_2'].append(adress.split(',')[1])

        else:

            sqldict['Address_2'].append('')



        # sqldict['Typology'].append(row['Insurance Types'])

        sqldict['Zip'].append(str(adress.split(' ')[-1]).replace(phone, '')[:5])

        sqldict['City'].append(adress.split(' ')[-3] if len(adress) > 0 else '' )

        sqldict['Phone'].append(phone)

        sqldict['RegulationType'].append('Licensed')

        sqldict['InternalID_1_type'].append('NAIC Company Code')

        sqldict['InternalID_1'].append(row['Cocode'])

        sqldict["Cntry"].append("US")  

        sqldict['ListProcessDate'].append(processdate)

        sqldict['RegCtry'].append(reg.split(' ')[0])

        sqldict['RegCode'].append(reg.split(' ')[1])

        sqldict['ListCode'].append(reg.split(' ')[-1])



    sqldict = bourange_same_length_array(sqldict)

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))




[INFO] : Working 1/1 _(US NAIC 1)_ 
[INFO] : - Clisk download Bouton
[INFO] : Download Excel file ... (wait 0/20 s)
[INFO] : Download Excel file ... (wait 2/20 s)
[INFO] : Download Excel file ... (wait 4/20 s)
[INFO] : Download Excel file ... (wait 6/20 s)
[INFO] : Download Excel file ... (wait 8/20 s)
[INFO] : Excel file = ['Company Results Phone.xlsx'])
[INFO] : - DataFrame 'Company Results Phone.xlsx' | containe = (5246, 8)


In [ ]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(filename, index=False)


driver.quit()

sleep(3)
    

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_6712\1466792167.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'XlsxWriter' object has no attribute 'save'

In [7]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,1st Atlantic Surety Co,15267,NAIC Company Code,,,...,,,,,,,,,,
1,,,,,,1st Auto & Cas Ins Co,44725,NAIC Company Code,,,...,,,,,,,,,,
2,,,,,,1st Choice Advantage Ins Co Inc,10750,NAIC Company Code,,,...,,,,,,,,,,
3,,,,,,4 Ever Life Ins Co,80985,NAIC Company Code,,,...,,,,,,,,,,
4,,,,,,5 Star Life Ins Co,77879,NAIC Company Code,,,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5241,,,,,,ZSI Ins Co RRG,17819,NAIC Company Code,,,...,,,,,,,,,,
5242,,,,,,Zurich Amer Ins Co,16535,NAIC Company Code,,,...,,,,,,,,,,
5243,,,,,,Zurich Amer Ins Co Of IL,27855,NAIC Company Code,,,...,,,,,,,,,,
5244,,,,,,Zurich Amer Life Ins Co,90557,NAIC Company Code,,,...,,,,,,,,,,
